# Scraper de datos musicales
Scraping de letras (AZLyrics) y géneros (Last.fm) a partir de un CSV propio.


In [1]:
# Ejecuta esta celda solo la primera vez
%pip install cloudscraper beautifulsoup4 pandas requests

Note: you may need to restart the kernel to use updated packages.


## Imports y configuración

In [14]:
import re
import time
import random
import logging
import pandas as pd
import cloudscraper
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import quote

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# Cloudscraper (bypassea Cloudflare en AZLyrics)
SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

# Cabeceras Last.fm 
HEADERS_LASTFM = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

# Delays entre requests
AZLYRICS_DELAY = (5, 10)   # AZLyrics tiene rate-limiting agresivo
LASTFM_DELAY   = (1, 3)

## Funciones de scraping

In [15]:
# "The Beatles" -> "thebeatles" para usarlo en la URL
def _slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())

def _get_lastfm(url: str):
    time.sleep(random.uniform(*LASTFM_DELAY))
    try:
        resp = SCRAPER.get(url, headers=HEADERS_LASTFM, timeout=15)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"Last.fm request failed: {url} → {e}")
        return None
    
def _get_azlyrics(url: str):
    time.sleep(random.uniform(*AZLYRICS_DELAY))
    try:
        resp = SCRAPER.get(url, timeout=20)
        resp.raise_for_status()
        return resp
    except Exception as e:
        log.warning(f"AZLyrics request failed: {url} → {e}")
        return None

def scrape_lyrics(artist: str, song: str):
    url = f"https://www.azlyrics.com/lyrics/{_slug(artist)}/{_slug(song)}.html"
    log.info(f"AZLyrics → {url}")
    resp = _get_azlyrics(url)
    if resp is None:
        return None
    if "ringtone" not in resp.text:
        log.warning(f"Página sin div.ringtone (canción no indexada)")
        return None
    soup = BeautifulSoup(resp.text, "html.parser")
    # Método 1: hermano de div.ringtone
    ringtone = soup.find("div", class_="ringtone")
    if ringtone:
        div = ringtone.find_next_sibling("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                log.info(f"Letra encontrada ({len(text)} chars)")
                return text
    # Método 2: comentario HTML
    for comment in soup.find_all(string=lambda t: isinstance(t, Comment) and "Usage of azlyrics" in t):
        div = comment.find_next("div")
        if div:
            text = div.get_text(separator="\n").strip()
            if len(text) > 80:
                return text
    # Método 3: div sin class largo
    for div in soup.find_all("div", class_=False, id=False):
        text = div.get_text(separator="\n").strip()
        if len(text) > 150 and text.count("\n") > 5:
            if not any(s in text[:60] for s in ["AZLyrics", "Submit", "Login", "©"]):
                return text
    log.warning(f"No se encontró letra para {artist} - {song}")
    return None

def _extract_lastfm_tags(html: str, top_n: int):
    soup = BeautifulSoup(html, "html.parser")
    tags = []
    for selector in [".catalogue-tags .tag", ".catalogue-tags a",
                     ".tag-list .tag", ".tags-list a", "ul.tags li a", "a.tag"]:
        for el in soup.select(selector):
            tag = el.get_text(strip=True).lower()
            if tag and tag not in tags and len(tag) > 1:
                tags.append(tag)
        if tags:
            break
    if not tags:
        seen = set()
        for a in soup.find_all("a", href=True):
            if "/tag/" in a["href"]:
                tag = a.get_text(strip=True).lower()
                if tag and tag not in seen and len(tag) > 1:
                    tags.append(tag); seen.add(tag)
    return tags[:top_n]

def scrape_genre(artist: str, song: str = None, top_n: int = 3):
    tags = []
    if song:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}/_/{quote(song)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    if not tags:
        resp = _get_lastfm(f"https://www.last.fm/music/{quote(artist)}")
        if resp:
            tags = _extract_lastfm_tags(resp.text, top_n)
    log.info(f"Géneros: {tags}")
    return tags

def scrape_dataset(songs: list, output_csv: str = "music_dataset.csv") -> pd.DataFrame:
    results = []
    for i, entry in enumerate(songs, 1):
        artist, song = entry["artist"], entry["song"]
        log.info(f"[{i}/{len(songs)}] {artist} — {song}")
        lyrics = scrape_lyrics(artist, song)
        genres = scrape_genre(artist, song, top_n=3)
        results.append({
            "artist":  artist, "song": song, "lyrics": lyrics,
            "genres":  ", ".join(genres) if genres else None,
            "genre_1": genres[0] if len(genres) > 0 else None,
            "genre_2": genres[1] if len(genres) > 1 else None,
            "genre_3": genres[2] if len(genres) > 2 else None,
        })
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8")
    log.info(f"Dataset guardado: {output_csv} ({len(results)} canciones)")
    return pd.DataFrame(results)

print("Funciones cargadas")


Funciones cargadas


## Cargar y unir CSVs

In [ ]:
# Hace falta cambiar estos datasets por unos con más canciones y solo letras en inglés o al menos en AZLyrics
# De momento hay que descargar estos desde "https://www.kaggle.com/datasets/neisse/scrapped-lyrics-from-6-genres?select=lyrics-data.csv"
songs_df = pd.read_csv("lyrics-data.csv")
artists_df = pd.read_csv("artists-data.csv")

print("lyrics-data.csv columnas:", songs_df.columns.tolist())
print("artists-data.csv columnas:", artists_df.columns.tolist())

# Join por ALink (songs) <-> Link (artists)
merged = songs_df.merge(artists_df[["Link", "Artist"]], left_on="ALink", right_on="Link", how="left")

# Filtrar inglés y construir lista
songs = [
    {"artist": row["Artist"].strip(), "song": row["SName"].strip()}
    for _, row in merged.iterrows()
    if pd.notna(row["SName"]) and pd.notna(row["Artist"])
    and row.get("language") == "en"
][:100] #iterar cada 100 canciones

print(f"\nCanciones en inglés listas para scrapear: {len(songs)}")
songs[:3]  # preview


lyrics-data.csv columnas: ['ALink', 'SName', 'SLink', 'Lyric', 'language']
artists-data.csv columnas: ['Artist', 'Genres', 'Songs', 'Popularity', 'Link']

Canciones en inglés listas para scrapear: 1000


[{'artist': 'Ivete Sangalo', 'song': 'Careless Whisper'},
 {'artist': 'Ivete Sangalo',
  'song': 'Could You Be Loved / Citação Musical do Rap: Se Ligue'},
 {'artist': 'Ivete Sangalo', 'song': "Cruisin' (Part. Saulo)"}]

## Ejecutar scraping
Con 1000 canciones este proceso tarda **3-5 horas**. El CSV se guarda tras cada canción,
así que puedes interrumpirlo y usar la celda de *Reanudar* para continuar.


In [17]:
OUTPUT_CSV = "music_dataset.csv"

df = scrape_dataset(songs, output_csv=OUTPUT_CSV)

print(f"Total canciones: {len(df)}")
print(f"Con letra:       {df['lyrics'].notna().sum()}")
print(f"Con géneros:     {df['genres'].notna().sum()}")


2026-04-24 12:24:34,254 [INFO] [1/1000] Ivete Sangalo — Careless Whisper
2026-04-24 12:24:34,255 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/ivetesangalo/carelesswhisper.html
2026-04-24 12:24:39,545 [WARNING] AZLyrics request failed: https://www.azlyrics.com/lyrics/ivetesangalo/carelesswhisper.html → 404 Client Error: Not Found for url: https://www.azlyrics.com/lyrics/ivetesangalo/carelesswhisper.html
2026-04-24 12:24:41,202 [INFO] Géneros: ['axe', 'female vocalists', 'brazilian']
2026-04-24 12:24:41,207 [INFO] [2/1000] Ivete Sangalo — Could You Be Loved / Citação Musical do Rap: Se Ligue
2026-04-24 12:24:41,208 [INFO] AZLyrics → https://www.azlyrics.com/lyrics/ivetesangalo/couldyoubelovedcitaomusicaldorapseligue.html


KeyboardInterrupt: 

## Reanudar scraping interrumpido

In [ ]:
import os

OUTPUT_CSV = "music_dataset.csv"

if os.path.exists(OUTPUT_CSV):
    done_df   = pd.read_csv(OUTPUT_CSV)
    done_keys = set(zip(done_df["artist"].str.lower(), done_df["song"].str.lower()))
    pending   = [s for s in songs if (s["artist"].lower(), s["song"].lower()) not in done_keys]
    print(f"Ya procesadas: {len(done_df)} | Pendientes: {len(pending)}")

    if pending:
        new_df   = scrape_dataset(pending, output_csv="music_dataset_new.csv")
        combined = pd.concat([done_df, new_df], ignore_index=True)
        combined.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
        print(f"Dataset combinado: {len(combined)} canciones")
    else:
        print("Todas las canciones ya están procesadas")
else:
    print("No existe music_dataset.csv, ejecuta primero la celda de scraping")


## Explorar resultados

In [18]:
df = pd.read_csv("music_dataset.csv")

print(f"Total canciones:  {len(df)}")
print(f"Con letra:        {df['lyrics'].notna().sum()} ({df['lyrics'].notna().mean():.0%})")
print(f"Con géneros:      {df['genres'].notna().sum()} ({df['genres'].notna().mean():.0%})")
print(f"\nGéneros más frecuentes:")
print(df["genre_1"].value_counts().head(10).to_string())


Total canciones:  1
Con letra:        0 (0%)
Con géneros:      1 (100%)

Géneros más frecuentes:
genre_1
axe    1


In [19]:
# Vista previa de las primeras filas
df[["artist", "song", "genres", "lyrics"]].head(10)

,artist,song,genres,lyrics
0,Ivete Sangalo,Careless Whisper,"axe, female vocalists, brazilian",NaN
